# Document OCR Pipeline — Complete Colab Walkthrough

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Document-OCR-Pipeline/blob/main/colab/document_ocr_pipeline.ipynb)

**Open in Google Colab:** https://colab.research.google.com/github/Gaurav14cs17/Document-OCR-Pipeline/blob/main/colab/document_ocr_pipeline.ipynb

Single notebook covering **every function** in `florence2.py` (and Qwen equivalent).

| Stage | Functions covered |
|-------|-------------------|
| **1** | `resolve_device`, `AutoProcessor`, tokenizer, `AutoModelForCausalLM`, `model.eval()` |
| **2** | `pad_info`, `processor()`, `model.generate()` |
| **3** | `batch_decode()` → raw text |
| **4** | `_parse_florence`, `post_process_generation`, `clean_florence_text` |
| **5** | `quad_to_bbox`, `unmap_bbox`, `_detect/_ocr/_layout/_table`, `task_result()` |

Set **`BACKEND`** and **`TASK`** in the config cell below.  
**GPU runtime** recommended (Runtime → Change runtime type → T4 GPU).

## 0 — Install & config

**Florence-2:** requires **`transformers==4.49.0`** (4.50+ breaks model loading).  
Run the install cell first. If you already imported `transformers` in this session, use **Runtime → Restart session**, then run all cells from the top.

In [ ]:
# Florence-2 breaks on transformers>=4.50 (forced_bos_token_id).
# Pin 4.49.0 and --upgrade so Colab downgrades from preinstalled 5.x.
%pip install -q --upgrade transformers==4.49.0
%pip install -q torch accelerate pillow qwen-vl-utils huggingface_hub bitsandbytes matplotlib requests

import transformers
print(f"transformers {transformers.__version__} (need <4.50 for Florence-2)")

In [ ]:
import json
import re
import tempfile
from pathlib import Path

import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import requests
from io import BytesIO
from PIL import Image, ImageDraw, ImageFont

# ── CONFIG ──────────────────────────────────────────────
BACKEND = "florence2"       # "florence2" (fast) | "qwen" (best quality)
TASK = "detect"             # detect | ocr | layout | table
RUN_ALL_TASKS = False       # True = run detect+ocr+layout+table at the end
MAX_NEW_TOKENS = 2048

MODEL_IDS = {
    "florence2": "microsoft/Florence-2-base-ft",
    "qwen": "Qwen/Qwen2.5-VL-3B-Instruct",
}
FLORENCE_PROMPTS = {
    "detect": "<OCR_WITH_REGION>",
    "ocr": "<OCR>",
    "layout": "<DENSE_REGION_CAPTION>",
    "table": "<OCR>",
}
QWEN_PROMPTS = {
    "layout": """Analyze this document page layout.
Return ONLY valid JSON (no markdown fences):
{"blocks": [{"label": "Title|Text|Table", "bbox": [x1,y1,x2,y2], "reading_order": 1}]}""",
    "ocr": "OCR this document page. Return clean reading-order text as HTML using <p> tags.",
    "table": """Find all tables. Return ONLY valid JSON:
{"tables": [{"bbox": [x1,y1,x2,y2], "rows": [["cell"]]}]}""",
    "detect": """Read all visible text lines. Return ONLY valid JSON:
{"text_lines": [{"text": "line", "bbox": [x1,y1,x2,y2], "confidence": 0.95}]}""",
}
HANDLERS_MAP = {
    "detect": "_detect  →  prompt <OCR_WITH_REGION>",
    "ocr": "_ocr  →  prompt <OCR>",
    "layout": "_layout  →  prompt <DENSE_REGION_CAPTION>",
    "table": "_table  →  calls _ocr internally",
}

MODEL_ID = MODEL_IDS[BACKEND]
PROMPT = FLORENCE_PROMPTS[TASK] if BACKEND == "florence2" else QWEN_PROMPTS[TASK]

def resolve_device():
    return "cuda" if torch.cuda.is_available() else "cpu"

DEVICE = resolve_device()
print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
print(f"Backend: {BACKEND}  |  Task: {TASK}  |  Model: {MODEL_ID}")
print(f"Handler: {HANDLERS_MAP[TASK]}")

---
## Stage 1 — `__init__`: load processor, tokenizer, model

Mirrors `Florence2Backend.__init__` in `florence2.py`:

1. **`resolve_device()`** — pick CPU or CUDA
2. **`AutoProcessor.from_pretrained()`** — tokenizer + image preprocessor
3. **`AutoModelForCausalLM.from_pretrained()`** — model weights
4. **`model.eval()`** — freeze for inference (no gradients)

In [ ]:
import transformers

_tf_major, _tf_minor = (int(x) for x in transformers.__version__.split(".")[:2])
if BACKEND == "florence2" and (_tf_major >= 5 or (_tf_major, _tf_minor) >= (4, 50)):
    raise ImportError(
        f"Florence-2 needs transformers<4.50 (you have {transformers.__version__}). "
        "Re-run the install cell, then Runtime → Restart session, and run all cells again."
    )

print("Step 1.1 — resolve_device()")
print(f"  → {DEVICE}")
print(f"  transformers {transformers.__version__}")

print("\nStep 1.2 — AutoProcessor.from_pretrained()")
if BACKEND == "florence2":
    from transformers import AutoProcessor, AutoModelForCausalLM
    processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
else:
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
    processor = AutoProcessor.from_pretrained(MODEL_ID)

tok = processor.tokenizer
print(f"  Processor class : {type(processor).__name__}")
print(f"  Tokenizer class : {type(tok).__name__}")
print(f"  Vocab size      : {tok.vocab_size}")
print(f"  Special tokens  : {tok.special_tokens_map}")

In [ ]:
print("Step 1.3 — Tokenizer demo (prompt → token IDs → decoded)")
sample = PROMPT if BACKEND == "florence2" else PROMPT[:80]
ids = tok.encode(sample)
print(f"  Prompt   : {sample[:100]}{'...' if len(sample)>100 else ''}")
print(f"  Token IDs: {ids[:12]}{'...' if len(ids)>12 else ''}")
print(f"  Count    : {len(ids)} tokens")
print(f"  Decoded  : {tok.decode(ids[:20])}")

In [ ]:
print("Step 1.4 — AutoModelForCausalLM.from_pretrained() + model.eval()")
if BACKEND == "florence2":
    dtype = torch.float16 if DEVICE == "cuda" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        torch_dtype=dtype,
        attn_implementation="eager",
    ).to(DEVICE)
else:
    kwargs = {}
    if DEVICE == "cuda":
        from transformers import BitsAndBytesConfig
        kwargs = {
            "device_map": "auto",
            "quantization_config": BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16
            ),
        }
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_ID, **kwargs)
    if DEVICE != "cuda":
        model = model.to(DEVICE)

model.eval()
params = sum(p.numel() for p in model.parameters())
print(f"  Model class : {type(model).__name__}")
print(f"  Parameters  : {params/1e6:.1f} M")
print(f"  Training?   : {model.training}  (False = eval mode ✓)")
print(f"  Device      : {next(model.parameters()).device}")

cfg = model.config
for k in ["model_type", "hidden_size", "num_hidden_layers", "vocab_size"]:
    if hasattr(cfg, k):
        print(f"  config.{k}: {getattr(cfg, k)}")

---
## Helper functions (from `vlm_pipeline/utils/`)

These are used across stages 2–5.

In [ ]:
def pad_info(image):
    """pad_info(image) — pad to square, return offsets for unmap_bbox."""
    w, h = image.size
    side = max(w, h)
    canvas = Image.new("RGB", (side, side), "white")
    pad_x, pad_y = (side - w) // 2, (side - h) // 2
    canvas.paste(image, (pad_x, pad_y))
    return canvas, pad_x, pad_y, w, h

def clean_html_tags(text):
    return re.sub(r"</?[a-zA-Z_][^>]*>", "", text).strip()

def clean_florence_text(text):
    return clean_html_tags(text)

def clean_model_response(text):
    text = text.strip()
    if "assistant" in text:
        text = text.split("assistant", 1)[-1].strip()
    fence = re.search(r"```(?:html|json)?\s*([\s\S]*?)\s*```", text)
    return fence.group(1).strip() if fence else text

def extract_json(text):
    text = clean_model_response(text)
    fence = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", text)
    if fence:
        text = fence.group(1).strip()
    return json.loads(text)

def quad_to_bbox(quad):
    xs, ys = quad[0::2], quad[1::2]
    return [int(min(xs)), int(min(ys)), int(max(xs)), int(max(ys))]

def unmap_bbox(bbox, pad_x, pad_y, orig_w, orig_h):
    x1, y1, x2, y2 = bbox
    x1 = max(0, min(orig_w, x1 - pad_x)); y1 = max(0, min(orig_h, y1 - pad_y))
    x2 = max(0, min(orig_w, x2 - pad_x)); y2 = max(0, min(orig_h, y2 - pad_y))
    return [] if x2 <= x1 or y2 <= y1 else [x1, y1, x2, y2]

def task_result(task, backend, **payload):
    return {"task": task, "backend": backend, **payload}

def parse_florence(processor, task_prompt, generated_text, pad_x, pad_y, orig_w, orig_h):
    """_parse_florence() — calls post_process_generation."""
    side = max(orig_w, orig_h)
    parsed = processor.post_process_generation(
        generated_text, task=task_prompt, image_size=(side, side)
    )
    return parsed, pad_x, pad_y, orig_w, orig_h

print("Helpers loaded: pad_info, quad_to_bbox, unmap_bbox, clean_florence_text, parse_florence, task_result")

---
## Load document image

Upload your own file or use the bundled sample page.

In [ ]:
# Option A — upload (uncomment in Colab)
# from google.colab import files
# uploaded = files.upload()
# image = Image.open(list(uploaded.keys())[0]).convert("RGB")

# Option B — sample from repo
try:
    url = "https://raw.githubusercontent.com/Gaurav14cs17/Document-OCR-Pipeline/main/assets/table_page.png"
    image = Image.open(BytesIO(requests.get(url, timeout=30).content)).convert("RGB")
except Exception:
    image = Image.new("RGB", (640, 480), "white")
    ImageDraw.Draw(image).text((20, 20), "Sample document", fill="black")

print(f"Image size: {image.size[0]} x {image.size[1]} px")
plt.figure(figsize=(8, 6)); plt.imshow(image); plt.title("Input document page"); plt.axis("off"); plt.show()

---
## Stage 2 — `_generate()` steps A→C: preprocess + inference

| Step | Function | Output |
|------|----------|--------|
| **A** | `pad_info(image)` | square image + `pad_x, pad_y, orig_w, orig_h` |
| **B** | `processor(text, images)` | `input_ids`, `pixel_values` tensors |
| **C** | `model.generate(...)` | output **token ID** tensor |

In [ ]:
pad_meta = {}

if BACKEND == "florence2":
    print("Step 2A — pad_info(image)")
    padded, pad_x, pad_y, orig_w, orig_h = pad_info(image)
    pad_meta = dict(pad_x=pad_x, pad_y=pad_y, orig_w=orig_w, orig_h=orig_h)
    print(f"  Original : {orig_w} x {orig_h}")
    print(f"  Padded   : {padded.size[0]} x {padded.size[1]} (square)")
    print(f"  pad_x={pad_x}, pad_y={pad_y}")

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].imshow(image); ax[0].set_title("Original"); ax[0].axis("off")
    ax[1].imshow(padded); ax[1].set_title(f"Padded square ({padded.size[0]}px)"); ax[1].axis("off")
    rect = patches.Rectangle((pad_x, pad_y), orig_w, orig_h, linewidth=2, edgecolor="red", facecolor="none")
    ax[1].add_patch(rect); ax[1].set_title("Padded (red = original area)"); plt.show()

    print(f"\nStep 2B — processor(text='{PROMPT}', images=padded)")
    inputs = processor(text=PROMPT, images=padded, return_tensors="pt").to(DEVICE)
    print(f"  input_ids     shape: {tuple(inputs['input_ids'].shape)}")
    print(f"  pixel_values  shape: {tuple(inputs['pixel_values'].shape)}")
    print(f"  input_ids[0][:15]: {inputs['input_ids'][0][:15].tolist()}")

    print(f"\nStep 2C — model.generate(max_new_tokens={MAX_NEW_TOKENS}, num_beams=1)")
    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=1,
            use_cache=False,
        )
else:
    from qwen_vl_utils import process_vision_info
    print("Step 2 — Qwen: apply_chat_template + process_vision_info + generate")
    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
        temp_path = tmp.name
    image.save(temp_path)
    messages = [{"role": "user", "content": [
        {"type": "image", "image": temp_path},
        {"type": "text", "text": PROMPT},
    ]}]
    chat_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    print(f"  Chat template length: {len(chat_text)} chars")
    print(f"  Preview: {chat_text[:200]}...")
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[chat_text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(model.device)
    print(f"  input_ids shape: {tuple(inputs['input_ids'].shape)}")
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)

print(f"\n  generated_ids shape : {tuple(generated_ids.shape)}")
print(f"  Total tokens        : {generated_ids.shape[1]}")
print(f"  First 20 token IDs  : {generated_ids[0][:20].tolist()}")
print(f"  Last 10 token IDs   : {generated_ids[0][-10:].tolist()}")

---
## Stage 3 — `_generate()` step D: `batch_decode()` → raw text

Florence-2 uses `skip_special_tokens=False` to keep structure for `post_process_generation`.

In [ ]:
print("Step 3 — processor.batch_decode(generated_ids)")
if BACKEND == "florence2":
    raw_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    raw_clean = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    print(f"  skip_special_tokens=False → {len(raw_text)} chars (used for parsing)")
    print(f"  skip_special_tokens=True  → {len(raw_clean)} chars (human readable)")
else:
    raw_text = processor.batch_decode(
        generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]
    raw_clean = raw_text

print("\n" + "="*70)
print("RAW TEXT (full model output)")
print("="*70)
print(raw_text[:2500])
if len(raw_text) > 2500:
    print(f"\n... [{len(raw_text)-2500} more characters]")

In [ ]:
fig = plt.figure(figsize=(15, 9))
ax1 = fig.add_subplot(2, 2, 1); ax1.imshow(image); ax1.set_title("① Input image"); ax1.axis("off")
ax2 = fig.add_subplot(2, 2, 2); ax2.axis("off")
ax2.text(0, 1, f"② Task: {TASK}\nBackend: {BACKEND}\n\nPrompt:\n{PROMPT[:400]}", va="top", fontsize=9, family="monospace")
ax2.set_title("Prompt sent to model")
ax3 = fig.add_subplot(2, 1, 2); ax3.axis("off")
ax3.text(0, 1, f"③ Raw decoded text ({len(raw_text)} chars):\n\n{raw_text[:1800]}", va="top", fontsize=8, family="monospace")
plt.tight_layout(); plt.show()

---
## Stage 4 — `_parse_florence` + `post_process_generation` + clean

```python
parsed = processor.post_process_generation(
    generated_text,
    task=task_prompt,
    image_size=(max(orig_w, orig_h), max(orig_w, orig_h)),
)
```

In [ ]:
print("Step 4 — _parse_florence / clean_model_response")

if BACKEND == "florence2":
    parsed, px, py, ow, oh = parse_florence(
        processor, PROMPT, raw_text,
        pad_meta["pad_x"], pad_meta["pad_y"],
        pad_meta["orig_w"], pad_meta["orig_h"],
    )
    region = parsed.get(PROMPT, {})
    print(f"  post_process_generation keys: {list(parsed.keys())}")
    if isinstance(region, dict):
        print(f"  Region keys: {list(region.keys())}")
        for k, v in region.items():
            if isinstance(v, list):
                print(f"    {k}: {len(v)} items  (first: {v[0] if v else 'empty'})")
            else:
                print(f"    {k}: {str(v)[:80]}")
    else:
        print(f"  Plain text region: {str(region)[:200]}")
else:
    cleaned = clean_model_response(raw_text)
    print(f"  clean_model_response: {len(cleaned)} chars")
    print(f"  Preview: {cleaned[:600]}")
    try:
        parsed = extract_json(cleaned)
        print(f"  extract_json keys: {list(parsed.keys()) if isinstance(parsed, dict) else type(parsed)}")
    except json.JSONDecodeError as e:
        parsed = {"raw": cleaned}
        print(f"  JSON parse failed: {e}")

In [ ]:
print("BEFORE vs AFTER cleaning")
print("-"*50)
print("BEFORE (raw_text):")
print(raw_text[:350])
print("\n" + "-"*50)
if BACKEND == "florence2":
    region = parsed.get(PROMPT, {})
    if isinstance(region, dict) and region.get("labels"):
        print("AFTER (clean_florence_text on first 3 labels):")
        for lbl in region["labels"][:3]:
            print(f"  raw: {lbl!r}  →  clean: {clean_florence_text(str(lbl))!r}")
    elif isinstance(region, str) or (isinstance(region, dict) is False):
        t = str(region if not isinstance(region, dict) else region)
        print(f"AFTER clean_florence_text: {clean_florence_text(t)[:300]}")
else:
    print(f"AFTER clean_model_response: {clean_model_response(raw_text)[:350]}")

---
## Stage 4b — `quad_to_bbox` + `unmap_bbox` demo (detect/layout)

Shows coordinate transform from padded square back to original image.

In [ ]:
if BACKEND == "florence2" and TASK in ("detect", "layout"):
    region = parsed.get(PROMPT, {})
    px, py, ow, oh = pad_meta["pad_x"], pad_meta["pad_y"], pad_meta["orig_w"], pad_meta["orig_h"]

    if TASK == "detect" and region.get("quad_boxes"):
        quad = region["quad_boxes"][0]
        label = region["labels"][0]
        bbox_padded = quad_to_bbox(quad)
        bbox_orig = unmap_bbox(bbox_padded, px, py, ow, oh)
        print(f"Demo line 1: {clean_florence_text(str(label))}")
        print(f"  quad (8 pts)     : {quad}")
        print(f"  quad_to_bbox     : {bbox_padded}  (on padded image)")
        print(f"  unmap_bbox       : {bbox_orig}  (on original image)")

        vis = image.copy(); draw = ImageDraw.Draw(vis)
        draw.rectangle(bbox_orig, outline="lime", width=3)
        draw.text((bbox_orig[0], max(0,bbox_orig[1]-14)), clean_florence_text(str(label))[:30], fill="lime")
        plt.figure(figsize=(8,6)); plt.imshow(vis); plt.title("First detected line — bbox on original image"); plt.axis("off"); plt.show()
    elif TASK == "layout" and region.get("bboxes"):
        bbox = [int(v) for v in region["bboxes"][0]]
        mapped = unmap_bbox(bbox, px, py, ow, oh)
        print(f"Demo block 1: {clean_florence_text(str(region['labels'][0]))[:60]}")
        print(f"  bbox on padded : {bbox}")
        print(f"  unmap_bbox     : {mapped}")
else:
    print("(quad_to_bbox / unmap_bbox demo skipped — set BACKEND=florence2, TASK=detect or layout)")

---
## Stage 5 — Task handlers → `task_result()` JSON

Mirrors `_detect`, `_ocr`, `_layout`, `_table` + `_dispatch` from `florence2.py`.

In [ ]:
def run_florence_task(task, image, processor, model, device, max_tokens=2048):
    """Full florence2.py handler for one task."""
    prompt = FLORENCE_PROMPTS[task]
    tokens = 1024 if task == "layout" else max_tokens
    padded, px, py, ow, oh = pad_info(image)
    inputs = processor(text=prompt, images=padded, return_tensors="pt").to(device)
    with torch.no_grad():
        gen = model.generate(
            input_ids=inputs["input_ids"], pixel_values=inputs["pixel_values"],
            max_new_tokens=tokens, num_beams=1, use_cache=False,
        )
    raw = processor.batch_decode(gen, skip_special_tokens=False)[0]
    parsed, px, py, ow, oh = parse_florence(processor, prompt, raw, px, py, ow, oh)

    if task == "detect":
        region = parsed.get(prompt, {})
        lines = []
        for quad, label in zip(region.get("quad_boxes", []), region.get("labels", [])):
            bbox = unmap_bbox(quad_to_bbox(quad), px, py, ow, oh)
            if bbox:
                lines.append({"text": clean_florence_text(label), "bbox": bbox, "confidence": 1.0})
        return task_result("detect", "florence2", text_lines=lines)
    if task == "ocr":
        text = parsed.get(prompt, "")
        text = clean_florence_text(str(text) if not isinstance(text, str) else text)
        return task_result("ocr", "florence2", text=text, html=f"<p>{text}</p>")
    if task == "layout":
        labels = parsed.get(prompt, {}).get("labels", [])
        bboxes = parsed.get(prompt, {}).get("bboxes", [])
        blocks = []
        for i, (bbox, label) in enumerate(zip(bboxes, labels), start=1):
            mapped = unmap_bbox([int(v) for v in bbox], px, py, ow, oh)
            if mapped:
                blocks.append({"label": "Text", "bbox": mapped, "reading_order": i,
                               "caption": clean_florence_text(str(label))})
        return task_result("layout", "florence2", blocks=blocks)
    ocr = run_florence_task("ocr", image, processor, model, device)
    return task_result("table", "florence2", tables=[], note="Use qwen for tables",
                       fallback_text=ocr.get("text", ""))

print(f"Running handler: {HANDLERS_MAP[TASK]}")
if BACKEND == "florence2":
    result = run_florence_task(TASK, image, processor, model, DEVICE, MAX_NEW_TOKENS)
else:
    if TASK == "ocr":
        result = task_result("ocr", "qwen", text=cleaned, html=cleaned)
    else:
        result = task_result(TASK, "qwen")
        result.update(parsed if isinstance(parsed, dict) else {"raw": parsed})

print(json.dumps(result, indent=2, ensure_ascii=False))

In [ ]:
def draw_result(image, result, task):
    vis = image.copy(); draw = ImageDraw.Draw(vis)
    if task == "detect" and result.get("text_lines"):
        for line in result["text_lines"]:
            b = line["bbox"]
            draw.rectangle(b, outline="red", width=2)
            draw.text((b[0], max(0, b[1]-12)), line["text"][:32], fill="red")
        return vis, f"detect — {len(result['text_lines'])} lines", "red"
    if task == "layout" and result.get("blocks"):
        for block in result["blocks"]:
            b = block["bbox"]
            draw.rectangle(b, outline="blue", width=2)
            draw.text((b[0], b[1]), f"#{block['reading_order']}", fill="blue")
        return vis, f"layout — {len(result['blocks'])} blocks", "blue"
    return vis, task, "black"

if TASK == "ocr":
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    axes[0].imshow(image); axes[0].set_title("Input"); axes[0].axis("off")
    axes[1].axis("off")
    axes[1].text(0, 1, result.get("text", "")[:3000], va="top", fontsize=9, family="monospace")
    axes[1].set_title("OCR text output")
else:
    vis, title, _ = draw_result(image, result, TASK)
    fig, axes = plt.subplots(1, 2, figsize=(15, 7))
    axes[0].imshow(image); axes[0].set_title("Input"); axes[0].axis("off")
    axes[1].imshow(vis); axes[1].set_title(title); axes[1].axis("off")
plt.tight_layout(); plt.show()

---
## Bonus — Run ALL tasks (detect + ocr + layout + table)

Same as: `--task detect --task ocr --task layout --task table`

In [ ]:
if RUN_ALL_TASKS and BACKEND == "florence2":
    all_results = {}
    for t in ["detect", "ocr", "layout", "table"]:
        print(f"Running {t}...")
        all_results[t] = run_florence_task(t, image, processor, model, DEVICE)
    print("\n" + "="*60)
    for t, res in all_results.items():
        print(f"\n--- {t.upper()} ---")
        if t == "detect":
            print(f"  {len(res.get('text_lines', []))} text lines")
        elif t == "ocr":
            print(f"  {len(res.get('text', ''))} chars of text")
        elif t == "layout":
            print(f"  {len(res.get('blocks', []))} layout blocks")
        else:
            print(f"  tables: {res.get('tables', [])}  note: {res.get('note', '')}")

    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    for ax, t in zip(axes.flat, ["detect", "ocr", "layout", "table"]):
        if t == "ocr":
            ax.imshow(image); ax.set_title(f"ocr: {len(all_results[t].get('text',''))} chars"); ax.axis("off")
        elif t in ("detect", "layout"):
            vis, title, _ = draw_result(image, all_results[t], t)
            ax.imshow(vis); ax.set_title(title); ax.axis("off")
        else:
            ax.imshow(image); ax.set_title("table (fallback OCR)"); ax.axis("off")
    plt.tight_layout(); plt.show()
else:
    print("Set RUN_ALL_TASKS = True and BACKEND = 'florence2' to run all four tasks.")

---
## Full pipeline map (all functions)

```
run_task(image, task)                    ← entry point
  └─ _dispatch → _handlers()[task]
       └─ _generate(image, task_prompt)
            A. pad_info(image)           → padded, pad_x, pad_y, orig_w, orig_h
            B. processor(text, images)   → input_ids, pixel_values
            C. model.generate(...)       → generated_ids
            D. batch_decode(...)         → raw_text
       └─ _parse_florence(...)
            post_process_generation(...) → parsed dict
       └─ quad_to_bbox / unmap_bbox     → original image coords
       └─ clean_florence_text           → clean strings
       └─ task_result(...)              → final JSON
```

CLI equivalent:
```bash
python -m vlm_pipeline assets/table_page.png \
  --backend florence2 --task detect --task ocr --page 0
```